# ResNet WeightTracker vs Physical Pruning

This uses the shared diagnostic helper:

1. clone the model twice
2. prune one clone physically with Torch-Pruning `BasePruner`
3. fake-prune the other clone by zeroing the same pruning groups
4. count the physical model with `fvcore.nn.FlopCountAnalysis`
5. count the zeroed model with WeightTracker active MAC calculations, not BOPs
6. print one table comparing axes and MACs per layer


In [1]:
import os
import sys
from pathlib import Path

import torch
from torchvision.models import resnet18

ROOT = Path("/Users/mikkeldahl/codeq_research/CoDeQ")
os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from sanity_checks.pruning_weighttracker_compare import compare_weighttracker_to_physical_pruning

torch.manual_seed(0)


/Users/mikkeldahl/codeq_research/CoDeQ/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Ple
ase update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = resnet18(weights=None).eval()
example_inputs = torch.randn(1, 3, 224, 224)

print(model(example_inputs).shape)


torch.Size([1, 1000])


In [3]:
# Keep the classifier output dimension fixed, as in the earlier playground.
ignored_layers = [model.fc]

result = compare_weighttracker_to_physical_pruning(
    model,
    example_inputs,
    pruning_ratio=0.5,
    ignored_layers=ignored_layers,
    round_to=8,
)


Physical fvcore totals
  physical total MACs: 485646080.0
  physical params: 3055880
Physical pruned model, fvcore by_module()
  module                 kind         physical_axes  physical_macs  params
  ---------------------  -----------  -------------  -------------  ---------
  conv1                  Conv2d       [3, 32]        5.901e+07      4704
  bn1                    BatchNorm2d  [32]           8.028e+05      64
  layer1.0.conv1         Conv2d       [32, 32]       2.89e+07       9216
  layer1.0.bn1           BatchNorm2d  [32]           2.007e+05      64
  layer1.0.conv2         Conv2d       [32, 32]       2.89e+07       9216
  layer1.0.bn2           BatchNorm2d  [32]           2.007e+05      64
  layer1.1.conv1         Conv2d       [32, 32]       2.89e+07       9216
  layer1.1.bn1           BatchNorm2d  [32]           2.007e+05      64
  layer1.1.conv2         Conv2d       [32, 32]       2.89e+07       9216
  layer1.1.bn2           BatchNorm2d  [32]           2.007e+05      64


`result` contains `zeroed_model`, `physical_model`, `tracker`, `physical_analysis`, `comparison_rows`, and `summary` for manual inspection.
